In [21]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 Quantum Key Distribution (No Attacker)

### Protocol overview
1. Alice picks random bits and random bases (rectilinear `+` or diagonal `×`), encodes each bit as a qubit.
2. Bob picks random bases and measures each qubit.
3. Alice and Bob publicly compare bases; they keep only the bits where they chose the same basis.
4. They sacrifice a small sample of the sifted key to check for errors. With no attacker the error rate should be 0.

In [22]:
%pip install qiskit==1.2.4 qiskit-aer==0.15.1 pylatexenc==2.10 -q

In [23]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import math

# Shared simulator
simulator = AerSimulator()

# Measures n qubits each prepared in |+⟩ = H|0⟩ to get n unbiased random bits.
def quantum_random_bits(n: int) -> list[int]:
    MAX_BATCH = 29
    bits = []
    remaining = n
    while remaining > 0:
        batch = min(remaining, MAX_BATCH)
        qc = QuantumCircuit(batch, batch)
        qc.h(range(batch))
        qc.measure(range(batch), range(batch))
        job = simulator.run(transpile(qc, simulator), shots=1, memory=True)
        result_str = job.result().get_memory()[0]
        bits.extend(int(b) for b in reversed(result_str))
        remaining -= batch
    return bits[:n]

print("Utility ready. Sample of 8 quantum random bits:", quantum_random_bits(8))

Utility ready. Sample of 8 quantum random bits: [1, 1, 1, 0, 0, 1, 1, 1]


In [24]:
# Alice encoding
# Basis convention: 0 → rectilinear (+), 1 → diagonal (×)
#   basis 0, bit 0 → |0⟩
#   basis 0, bit 1 → |1⟩
#   basis 1, bit 0 → |+⟩  (H|0⟩)
#   basis 1, bit 1 → |−⟩  (H|1⟩)
N_QUBITS = 100
SAMPLE_FRACTION = 0.2

def alice_encode(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

# Alice generate her secret bits and random bases
alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)
alice_circuits = alice_encode(alice_bits, alice_bases)

print(f"Alice prepared {N_QUBITS} qubits.")
print(f"Alice bits (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]} (0=+, 1=×)")

Alice prepared 100 qubits.
Alice bits (first 20): [1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1]
Alice bases (first 20): [1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0] (0=+, 1=×)


In [25]:
# Bob measurement
# Bob independently picks random bases and measures each qubit Alice sent.
def bob_measure(circuits: list[QuantumCircuit], bases: list[int]) -> list[int]:
    results = []
    for qc, basis in zip(circuits, bases):
        meas_qc = qc.copy()
        if basis == 1:
            meas_qc.h(0) # rotate back from diagonal basis before measuring
        meas_qc.measure(0, 0)
        job = simulator.run(transpile(meas_qc, simulator), shots=1, memory=True)
        bit = int(job.result().get_memory()[0])
        results.append(bit)
    return results

bob_bases = quantum_random_bits(N_QUBITS)
bob_bits  = bob_measure(alice_circuits, bob_bases)

print(f"Bob measured {N_QUBITS} qubits.")
print(f"Bob bases (first 20): {bob_bases[:20]} (0=+, 1=×)")
print(f"Bob bits (first 20): {bob_bits[:20]}")

Bob measured 100 qubits.
Bob bases (first 20): [1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0] (0=+, 1=×)
Bob bits (first 20): [1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1]


In [26]:
# Sifting Alice and Bob publicly compare bases
def sift_key(alice_bases, bob_bases, alice_bits, bob_bits):
    alice_sifted, bob_sifted, positions = [], [], []
    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])
            positions.append(i)
    return alice_sifted, bob_sifted, positions

alice_sifted, bob_sifted, matching_positions = sift_key(
    alice_bases, bob_bases, alice_bits, bob_bits
)

print(f"Sifted key length: {len(alice_sifted)} (expected ≈ {N_QUBITS//2})")
print(f"Alice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob sifted (first 20): {bob_sifted[:20]}")

Sifted key length: 51 (expected ≈ 50)
Alice sifted (first 20): [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]
Bob sifted (first 20): [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]


In [27]:
# Error checking
DETECTION_THRESHOLD = 0.10 # flag an attack if error rate exceeds 10%

sample_size = max(1, int(len(alice_sifted) * SAMPLE_FRACTION))
sample_alice = alice_sifted[:sample_size]
sample_bob   = bob_sifted[:sample_size]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

print("   Error checking ──────────────────────────────────")
print(f"  Sample size : {sample_size} bits")
print(f"  Errors found: {errors}")
print(f"  Error rate : {error_rate:.1%}")
print(f"  Threshold : {DETECTION_THRESHOLD:.0%}")

if error_rate > DETECTION_THRESHOLD:
    print("  Attack detected")
else:
    print("  No attack detected.")
    final_key_alice = alice_sifted[sample_size:]
    final_key_bob = bob_sifted[sample_size:]
    assert final_key_alice == final_key_bob, "Keys do not match"
    print(f"  Final key length: {len(final_key_alice)} bits")
    print(f"  Final key (first 20 bits): {final_key_alice[:20]}")

   Error checking ──────────────────────────────────
  Sample size : 10 bits
  Errors found: 0
  Error rate : 0.0%
  Threshold : 10%
  No attack detected.
  Final key length: 41 bits
  Final key (first 20 bits): [1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
